
# 13. raw-schedule 로드 확인. (External Location)
------

--------


### 0. 패키지 및 라이브러리 불러오기 

In [0]:
%pip install openpyxl html5lib lxml beautifulsoup4 xlrd fsspec

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 24.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import pandas as pd
import numpy as np
import hashlib
import json
import xml.etree.ElementTree as ET
from io import StringIO
from openpyxl import load_workbook

### 1. 파일 확장자별로 pandas가 잘 읽도록 바꿔주는 함수 실행

In [0]:
# ===========
# 1: 자격증명으로 경로 불러오기
# ===========
blob_path = "abfss://raw-schedule@dt4team3storage.dfs.core.windows.net/"

files = dbutils.fs.ls(blob_path)
for f in files:
    print(f.name, f.size)



terminal_1_schedule_20260625_10.xls 25420
terminal_2_schedule_20260625_17.xls 22458
terminal_3_schedule_20260625_17.xlsx 6766
terminal_4_schedule_20260625_11.xls 23944
terminal_4_schedule_20260625_17.xls 23946
terminal_5_schedule_20260625_11.xlsx 28218
terminal_6_schedule_20260625_17 (1).xml 24998
terminal_7_schedule_20260625_11.xlsx 44370


In [0]:
# =============
# 2: 함수 
# =============
def copy_blob_file_to_local(blob_file_path):
    '''
    Blob 엑셀을 Wordspace 로 복사하는 함수
    '''
    workspace_tmp_dir = "/Workspace/Users/4dt035@msacademy.msai.kr/tmp_excel"
    dbutils.fs.mkdirs(workspace_tmp_dir)
    
    local_path = f"{workspace_tmp_dir}/{os.path.basename(blob_file_path)}"
    dbutils.fs.cp(blob_file_path, f"file:{local_path}")

    return local_path


def get_excel_engine(file_path):
    '''
    확장자별 pandas engine 선택
    '''
    if file_path.endswith(".xlsx"):
        return "openpyxl"
    elif file_path.endswith(".xls"):
        return "xlrd"
    else:
        return None
    

def read_html_xls_safely(local_path):
    '''
    인코딩명 보정하여 html 파일 읽기
    '''
    with open(local_path, "rb") as f:
        content = f.read()

    # 잘못 적힌 인코딩명 보정
    content = content.replace(b"udf-8", b"utf-8")
    content = content.replace(b"UDF-8", b"UTF-8")

    try:
        html_text = content.decode("utf-8")
    except UnicodeDecodeError:
        html_text = content.decode("cp949", errors="replace")

    # 디버깅용: 진짜 html인지 확인
    # print(html_text[:200])

    tables = pd.read_html(StringIO(html_text))
    return tables

def read_xml_preview(local_path, nrows=10):
    '''
    6부두 xml 파일의 태그 읽기
    '''
    tree = ET.parse(local_path)
    root = tree.getroot()

    def clean(tag):
        return tag.split("}")[-1]

    rows = []

    for elem in root.iter():
        if clean(elem.tag) == "Row":
            row = {}

            for child in elem:
                col_name = child.attrib.get("id")
                value = child.text

                if col_name:
                    row[col_name] = value

            rows.append(row)

    raw = pd.DataFrame(rows).head(nrows)

    return [{
        "sheet": "xml_row",
        "shape_preview": raw.shape,
        "first_5_rows": raw.head(5).values.tolist()
    }]


def read_excel_or_html_or_xml_table(local_path, nrows=10):
    '''
    모든 확장자에 대하여 파일 읽기 함수
    '''
    file_name = os.path.basename(local_path)

    # xlsx는 진짜 엑셀이므로 openpyxl
    if file_name.endswith(".xlsx"):
        xls = pd.ExcelFile(local_path, engine="openpyxl")
        results = []

        for sheet in xls.sheet_names:
            raw = pd.read_excel(
                local_path,
                sheet_name=sheet,
                header=None,
                nrows=nrows,
                engine="openpyxl"
            )

            results.append({
                "sheet": sheet,
                "shape_preview": raw.shape,
                "first_5_rows": raw.head(5).values.tolist()
            })

        return results
      
    # xls는 HTML 표일 가능성이 있으므로 read_html
    elif file_name.endswith(".xls"):
        tables = read_html_xls_safely(local_path)
        results = []

        for i, table in enumerate(tables):
            raw = table.head(nrows)

            results.append({
                "sheet": f"html_table_{i}",
                "shape_preview": raw.shape,
                "first_5_rows": raw.head(5).values.tolist()
            })

        return results
        # xml 파일 읽기
    elif file_name.endswith(".xml"):
        return read_xml_preview(local_path, nrows)

### 3. 1,2,4부두(html) 임시컬럼 붙이기 / 6부두(xml) 파싱 위치 조정하기

In [0]:
# ============
# html인 1,2,4부두 확인
# ============
from io import StringIO# 1,2,4부두 .xls(html) 전체 확인


xls_paths = [f.path for f in files if f.name.endswith(".xls")]

html_results = []

for path in xls_paths:
    local_path = copy_blob_file_to_local(path)

    with open(local_path, "rb") as f:
        content = f.read()

    content = content.replace(b"udf-8", b"utf-8").replace(b"UDF-8", b"UTF-8")

    try:
        html_text = content.decode("utf-8")
    except UnicodeDecodeError:
        html_text = content.decode("cp949", errors="replace")

    tables = pd.read_html(StringIO(html_text))
    df = tables[0]
    df.columns = [f"col_{i+1}" for i in range(df.shape[1])]

    html_results.append({
        "file": os.path.basename(path),
        "rows": df.shape[0],
        "cols": df.shape[1],
        "columns": df.columns.tolist()
    })

    print(os.path.basename(path))
    display(df.head())
    print(df.shape)
    print("-" * 80)

summary_html_df = pd.DataFrame(html_results)
display(summary_html_df)

terminal_1_schedule_20260625_10.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15
T3(P),HLC,IQUX004,2623E/2623E,41 (49) 62,IQUIQUE EXPRESS,AN3E,2026-06-21 18:00,2026-06-22 05:30,2026-06-24 11:12,2151,2991,278,N,DEPARTED
T1(P),ZIM,ZSEM004,6S/6S,01 (13) 18,SEASMILE,PANDA,2026-06-22 00:00,2026-06-22 10:25,2026-06-24 08:00,1316,1276,322,N,DEPARTED
T2(P),ONE,OOSY003,2612W/2612W,22 (29) 42,ONE SERENITY,AN3W,2026-06-22 16:00,2026-06-23 02:10,2026-06-25 13:00,2443,1345,1016,N,ARRIVED
T1(P),MSC,MABX001,UK623A/UK623A,02 (16) 23,MSC ABY X,CHINKE,2026-06-24 00:00,2026-06-24 10:45,2026-06-25 23:00,981,1064,346,N,ARRIVED
T3(P),MSC,MREG002,UX623A/UX623A,41 (50) 63,MSC REGULUS,SANTA,2026-06-24 03:00,2026-06-24 14:00,2026-06-26 13:00,1096,2348,818,N,ARRIVED


(18, 15)
--------------------------------------------------------------------------------
terminal_2_schedule_20260625_17.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17
1,HUDSON EXPRESS,HUDX-004/2026,623W/626E,HLC,GUSWC5E,Port,2026-06-23 17:20,2026-06-25 02:25,B4,2026-06-23 01:01,1817,1312,82,New Port Marine,null,2026-06-23 16:44
2,MSC KATIE,MKTE-001/2026,GO624N/GO624N,MSC,ORIENT E,Port,2026-06-23 21:00,2026-06-25 05:12,B8,2026-06-23 08:00,775,1267,6,New Port Marine,null,2026-06-23 20:22
3,JPO LIBRA,MJBR-007/2026,625S/626N,MAE,A04,Port,2026-06-24 00:45,2026-06-25 10:00,B5,2026-06-23 13:00,1899,996,0,ENS MARINE,null,2026-06-23 23:02
4,GSL GRANIA,GGRN-004/2026,624E/624E,MAE,GUSWC3E,Port,2026-06-24 08:25,2026-06-25 15:00,B6,2026-06-23 21:00,1022,1032,454,ENS MARINE,null,2026-06-24 04:02
5,TEMA MAERSK,TEMM-001/2026,624E/624E,MAE,GUSEC3,Port,2026-06-24 15:15,2026-06-26 02:00,B7,2026-06-24 00:00,1209,2493,34,ENS MARINE,null,2026-06-24 08:02


(33, 17)
--------------------------------------------------------------------------------
terminal_4_schedule_20260625_11.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16
T2(S),ONE,OOST001,029E/029E,ONE STORK,EC2E,2026-06-19 00:00,2026-06-21 20:30,2026-06-22 08:30,2026-06-24 22:00,2026-06-19 01:00,1297,4061,350,N,DEPARTED
T3(S),HMM,HONR003,0019W/0019W,HMM NURI,PS6W,2026-06-19 00:00,2026-06-22 00:18,2026-06-22 12:18,2026-06-25 04:00,2026-06-19 20:00,2102,1937,660,N,DEPARTED
T1(S),ONE,O1DN012,0007N/0007N,ONE DANIELLA,JPHN,2026-06-20 00:00,2026-06-23 06:06,2026-06-23 18:06,2026-06-24 07:00,2026-06-21 04:00,200,238,0,N,DEPARTED
T1(S),HMM,HHMN006,0025W/0025W,HMM MANILA,AAD,2026-06-21 00:00,2026-06-23 20:30,2026-06-24 08:30,2026-06-25 00:00,2026-06-22 11:00,337,222,22,N,DEPARTED
T2(S),YML,YIPT001,0024E/0024E,YM TIPTOP,EC1E,2026-06-22 00:00,2026-06-24 12:00,2026-06-25 00:00,2026-06-26 21:00,2026-06-22 00:00,904,3202,0,N,ARRIVED


(19, 16)
--------------------------------------------------------------------------------
terminal_4_schedule_20260625_17.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16
T2(S),ONE,OOST001,029E/029E,ONE STORK,EC2E,2026-06-19 00:00,2026-06-21 20:30,2026-06-22 08:30,2026-06-24 22:00,2026-06-19 01:00,1297,4061,350,N,DEPARTED
T3(S),HMM,HONR003,0019W/0019W,HMM NURI,PS6W,2026-06-19 00:00,2026-06-22 00:18,2026-06-22 12:18,2026-06-25 04:00,2026-06-19 20:00,2102,1937,660,N,DEPARTED
T1(S),ONE,O1DN012,0007N/0007N,ONE DANIELLA,JPHN,2026-06-20 00:00,2026-06-23 06:06,2026-06-23 18:06,2026-06-24 07:00,2026-06-21 04:00,200,238,0,N,DEPARTED
T1(S),HMM,HHMN006,0025W/0025W,HMM MANILA,AAD,2026-06-21 00:00,2026-06-23 20:30,2026-06-24 08:30,2026-06-25 00:00,2026-06-22 11:00,337,222,22,N,DEPARTED
T2(S),YML,YIPT001,0024E/0024E,YM TIPTOP,EC1E,2026-06-22 00:00,2026-06-24 12:00,2026-06-25 00:00,2026-06-26 17:00,2026-06-22 00:00,904,3202,0,N,ARRIVED


(19, 16)
--------------------------------------------------------------------------------


file,rows,cols,columns
terminal_1_schedule_20260625_10.xls,18,15,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15)"
terminal_2_schedule_20260625_17.xls,33,17,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16, col_17)"
terminal_4_schedule_20260625_11.xls,19,16,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16)"
terminal_4_schedule_20260625_17.xls,19,16,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16)"


In [0]:
# ============
# xml인 6부두 원문확인
# ============

xml_files = [
    f for f in files
    if f.name.lower().endswith(".xml")
]

for f in xml_files:
    print(f.name, f.size, f.path)

xml_path = xml_files[0].path

xml_text_df = spark.read.text(xml_path)

display(xml_text_df.limit(30))

terminal_6_schedule_20260625_17 (1).xml 24998 abfss://raw-schedule@dt4team3storage.dfs.core.windows.net/terminal_6_schedule_20260625_17 (1).xml


value
"<?xml version=""1.0"" encoding=""UTF-8""?>"
""
""
0
""
""
""
""
""
""


In [0]:
# ==========
# 6번부두 xml 파싱 위치 조정
# ==========

import xml.etree.ElementTree as ET
import pandas as pd

xml_file = [f for f in files if f.name.lower().endswith(".xml")][0]
local_path = copy_blob_file_to_local(xml_file.path)

tree = ET.parse(local_path)
root = tree.getroot()

def clean(tag):
    return tag.split("}")[-1]

rows = []

for elem in root.iter():
    if clean(elem.tag) == "Row":
        row = {}

        for child in elem:
            col_name = child.attrib.get("id")
            value = child.text

            if col_name:
                row[col_name] = value

        rows.append(row)

xml_df = pd.DataFrame(rows)

display(xml_df.head())
print(xml_df.shape)
print(xml_df.columns.tolist())

plvVoy,plvShiftvan,cdvOperator,atbYn,atdYn,plvStatus,plvEvoyout,plvAtd,cdvName,plvAtb,plvVsl,plvLodvan,plvRoute,plvDisvan,plvYear,plvEvoyin,plvBerth,plvVslvoy,plvQuarantine,plvNeartml
001,234,ZIM,Y,Y,Departed,10E,2026-06-25 09:00,ZIM GEMINI,2026-06-24 05:00,ZZB3,631,ZCP,548,2026,10E,2(S),ZZB3001,null,null
002,6,MSC,Y,Y,Departed,FW625W,2026-06-25 17:00,MSC THAIS,2026-06-24 08:00,MTHA,854,SWAN,1584,2026,FW618E,1(S),MTHA002,검역,null
001,0,HAS,Y,Y,Departed,2630E,2026-06-25 03:00,PACIFIC TIANJIN,2026-06-24 17:00,PCTJ,90,SETO3,64,2026,2628W,3(S),PCTJ001,null,null
018,0,HAS,Y,Y,Departed,2625E,2026-06-25 11:00,PACIFIC NINGBO,2026-06-25 05:00,PCNB,83,JSW1,64,2026,2624W,3(S),PCNB018,null,null
001,14,MSC,Y,N,Working,FY626A,2026-06-26 17:00,MSC SVEVA,2026-06-25 13:00,MSEV,708,AFRICA,1685,2026,FY626A,2(S),MSEV001,검역,null


(35, 20)
['plvVoy', 'plvShiftvan', 'cdvOperator', 'atbYn', 'atdYn', 'plvStatus', 'plvEvoyout', 'plvAtd', 'cdvName', 'plvAtb', 'plvVsl', 'plvLodvan', 'plvRoute', 'plvDisvan', 'plvYear', 'plvEvoyin', 'plvBerth', 'plvVslvoy', 'plvQuarantine', 'plvNeartml']


### 4. pandas로 첫 5행 확인하기

In [0]:
# ===========
# 8: 판다스로 바로 읽기
# ===========
import pandas as pd

# 1단계에서 리스팅한 파일 경로들을 리스트로
file_paths = [f.path for f in files if f.name.endswith(('.xlsx', '.xls' , ".xml"))]

results = []

for path in file_paths:
    try:
        local_path = copy_blob_file_to_local(path)
        table_results = read_excel_or_html_or_xml_table(local_path, nrows=10)

        for table_result in table_results:
            results.append({
                "file": os.path.basename(path),
                "sheet": table_result["sheet"],
                "shape_preview": table_result["shape_preview"],
                "first_5_rows": table_result["first_5_rows"]
            })

    except Exception as e:
        results.append({
            "file": os.path.basename(path),
            "error": str(e)
        })

for r in results:
    print(r)
    print("-" * 80)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b8568412-a617-4ada-8c33-2897eb2dff49/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'file': 'terminal_1_schedule_20260625_10.xls', 'sheet': 'html_table_0', 'shape_preview': (10, 15), 'first_5_rows': [['T3(P)', 'HLC', 'IQUX004', '2623E/2623E', '41 (49) 62', 'IQUIQUE EXPRESS', 'AN3E', '2026-06-21 18:00', '2026-06-22 05:30', '2026-06-24 11:12', 2151, 2991, 278, 'N', 'DEPARTED'], ['T1(P)', 'ZIM', 'ZSEM004', '6S/6S', '01 (13) 18', 'SEASMILE', 'PANDA', '2026-06-22 00:00', '2026-06-22 10:25', '2026-06-24 08:00', 1316, 1276, 322, 'N', 'DEPARTED'], ['T2(P)', 'ONE', 'OOSY003', '2612W/2612W', '22 (29) 42', 'ONE SERENITY', 'AN3W', '2026-06-22 16:00', '2026-06-23 02:10', '2026-06-25 13:00', 2443, 1345, 1016, 'N', 'ARRIVED'], ['T1(P)', 'MSC', 'MABX001', 'UK623A/UK623A', '02 (16) 23', 'MSC ABY X', 'CHINKE', '2026-06-24 00:00', '2026-06-24 10:45', '2026-06-25 23:00', 981, 1064, 346, 'N', 'ARRIVED'], ['T3(P)', 'MSC', 'MREG002', 'UX623A/UX623A', '41 (50) 63', 'MSC REGULUS', 'SANTA', '2026-06-24 03:00', '2026-06-24 14:00', '2026-06-26 13:00', 1096, 2348, 818, 'N', 'ARRIVED']]}
--------